전처리 및 가상ID 처리

In [ ]:
import pandas as pd
from tqdm import tqdm

def run_kci_only_ai_preprocessing(node_file, edge_file):
    print("1/4. 데이터 로드 및 AI 시드(Seed) 논문 필터링...")
    df_nodes = pd.read_csv(node_file)
    
    # [조건] 2015년 이후 발간 & 제목 또는 초록에 키워드 포함
    # AI는 대문자만, 나머지는 대소문자 무관
    ai_pattern = r'AI|(?i:Artificial Intelligence|인공지능)'

    # mask_ai = (df_nodes['발행연도'] >= 2015) & (
    #     # 여기서 case=True로 설정해야 패턴 내의 개별 설정이 작동합니다.
    #     df_nodes['제목'].str.contains(ai_pattern, case=True, na=False) | 
    #     df_nodes['초록'].str.contains(ai_pattern, case=True, na=False)
    # )
    mask_ai = (
        # 여기서 case=True로 설정해야 패턴 내의 개별 설정이 작동합니다.
        df_nodes['제목'].str.contains(ai_pattern, case=True, na=False) | 
        df_nodes['초록'].str.contains(ai_pattern, case=True, na=False)
    )
    ai_seed_df = df_nodes[mask_ai].copy()
    ai_seed_ids = set(ai_seed_df['논문ID'].unique())
    all_kci_ids = set(df_nodes['논문ID'].unique()) # 전체 KCI 논문 ID 집합
    
    print(f"   -> AI 시드 논문 선정: {len(ai_seed_ids)}편")

    print("2/4. 인용 엣지 필터링 (KCI to KCI)...")
    df_edges = pd.read_csv(edge_file)
    
    # 필터링 규칙:
    # 1. 인용을 한 논문(source_id)이 AI 시드 논문이어야 함
    # 2. 인용을 당한 논문(target_arti_id)이 KCI ID를 가지고 있어야 함
    # 3. 그 target_arti_id가 실제로 노드 리스트(KCI)에 존재해야 함
    
    mask_kci_to_kci = (
        df_edges['source_id'].isin(ai_seed_ids) & 
        df_edges['target_arti_id'].notna() & 
        df_edges['target_arti_id'].isin(all_kci_ids)
    )
    
    final_edges = df_edges[mask_kci_to_kci].copy()
    # 컬럼명 통일
    final_edges = final_edges.rename(columns={'target_arti_id': 'target_id_final'})
    
    print(f"   -> 필터링된 KCI 간 인용 관계 수: {len(final_edges)}")

    print("3/4. 최종 분석 대상 노드 리스트 정리...")
    # 엣지에 등장하는 모든 논문 ID (Source + Target)|
    active_node_ids = set(final_edges['source_id'].unique()) | set(final_edges['target_id_final'].unique())
    
    # 노드 메타데이터 추출
    final_nodes = df_nodes[df_nodes['논문ID'].isin(active_node_ids)].copy()
    final_nodes = final_nodes.rename(columns={'논문ID': 'node_id'})
    
    print("4/4. KCI 전용 AI 데이터 저장 중...")
    # 최종 결과물 저장
    final_edges[['source_id', 'target_id_final', 'ref_type']].to_csv('KCI_AI_전용_인용_엣지.csv', index=False, encoding='utf-8-sig')
    final_nodes[['node_id', '발행연도', '제목', '저자', '학술지명']].to_csv('KCI_AI_전용_논문_노드.csv', index=False, encoding='utf-8-sig')

    print(f"\n✅ KCI 전용 정제 완료!")
    print(f"- 최종 노드 수: {len(final_nodes)}개")
    print(f"- 최종 엣지 수: {len(final_edges)}개")

if __name__ == "__main__":
    run_kci_only_ai_preprocessing('전체_법학_논문목록_정제본.csv', '법학_인용_네트워크_데이터_전체.csv')

In [ ]:
# 최종 정제된 노드 파일 로드
final_nodes = pd.read_csv('KCI_AI_전용_논문_노드.csv')

# 발행연도별 분포 확인
year_dist = final_nodes['발행연도'].value_counts().sort_index()
print("📊 최종 데이터에 포함된 논문의 발행연도 분포:")
print(year_dist.head(10)) # 아주 오래된 논문들도 포함되어 있을 것입니다.

# 2015년 이전 논문 비중 계산
pre_2015 = final_nodes[final_nodes['발행연도'] < 2015]
print(f"\n💡 2015년 이전 발행된 논문 수: {len(pre_2015)}개")

# 분절분석

In [1]:
import pandas as pd
import networkx as nx
from tqdm import tqdm

# 1. 데이터 로드
df_nodes = pd.read_csv('KCI_AI_전용_논문_노드.csv')
df_edges = pd.read_csv('KCI_AI_전용_인용_엣지.csv')

# 2. 분석을 위해 엣지 데이터에 Source 논문의 발행연도 매칭
df_edges = df_edges.merge(df_nodes[['node_id', '발행연도']], left_on='source_id', right_on='node_id', how='left')

def analyze_discrete_network(start_year=2015, end_year=2024):
    yearly_metrics = []
    
    for year in tqdm(range(start_year, end_year + 1), desc="연도별 지표 계산 중"):
        # --- [추가 코드] 해당 연도에 실제로 '발행'된 논문 숫자 계산 ---
        num_papers_published = len(df_nodes[df_nodes['발행연도'] == year])
        
        # 해당 연도에 발행된 논문들이 생성한 엣지만 추출
        year_edges = df_edges[df_edges['발행연도'] == year]
        
        if year_edges.empty:
            # 엣지가 없더라도 발행 논문 수는 기록하기 위해 처리
            yearly_metrics.append({
                '연도': year,
                '발행논문수': num_papers_published,
                '네트워크_노드수': 0,
                '엣지수': 0,
                '밀도': 0,
                '주요논문': "데이터 없음"
            })
            continue
            
        # 그래프 생성
        G = nx.DiGraph()
        G.add_edges_from(zip(year_edges['source_id'], year_edges['target_id_final']))
        
        # 지표 계산
        num_nodes = G.number_of_nodes() # 네트워크에 포함된 논문 수 (인용한 논문 + 인용된 논문)
        num_edges = G.number_of_edges()
        density = nx.density(G)
        
        # 인차수(In-degree) 상위 논문 추출
        in_degrees = dict(G.in_degree())
        top_nodes = sorted(in_degrees.items(), key=lambda x: x[1], reverse=True)[:3]
        
        top_titles = []
        for node_id, count in top_nodes:
            title_info = df_nodes[df_nodes['node_id'] == node_id]['제목']
            title = title_info.values[0] if not title_info.empty else "Unknown"
            top_titles.append(f"{title}({count}회)")
            
        yearly_metrics.append({
            '연도': year,
            '발행논문수': num_papers_published,  # 추가된 지표
            '네트워크_노드수': num_nodes,
            '엣지수': num_edges,
            '밀도': round(density, 5),
            '주요논문': ", ".join(top_titles)
        })
        
    return pd.DataFrame(yearly_metrics)

# 분석 실행
discrete_results = analyze_discrete_network()
discrete_results.to_csv('연도별_분절분석_결과.csv', index=False, encoding='utf-8-sig')

print("\n📊 연도별 주요 지표 요약 (발행논문수 포함):")
print(discrete_results[['연도', '발행논문수', '네트워크_노드수', '엣지수', '밀도']])

연도별 지표 계산 중: 100%|██████████| 10/10 [00:00<00:00, 184.00it/s]


📊 연도별 주요 지표 요약 (발행논문수 포함):
     연도  발행논문수  네트워크_노드수   엣지수       밀도
0  2015    286        79    68  0.01104
1  2016    368       218   224  0.00474
2  2017    430       453   527  0.00257
3  2018    462       581   732  0.00217
4  2019    435       646   763  0.00183
5  2020    473      1071  1394  0.00122
6  2021    462      1314  1677  0.00097
7  2022    425      1125  1455  0.00115
8  2023    378      1540  2043  0.00086
9  2024    386      1951  2738  0.00072


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# 1. 분석 결과 데이터 로드
df = pd.read_csv('연도별_분절분석_결과.csv')

# 2. 시각화 (이중 축 활용)
fig, ax1 = plt.subplots(figsize=(12, 7))

# 축 1: 발행논문수 및 네트워크 노드수 (Bar)
# 두 데이터를 나란히 보여주기 위해 바 차트를 겹치거나 나열할 수 있습니다.
ax1.bar(df['연도'] - 0.2, df['발행논문수'], width=0.4, color='lightgray', alpha=0.7, label='Published AI Papers')
ax1.bar(df['연도'] + 0.2, df['네트워크_노드수'], width=0.4, color='skyblue', alpha=0.6, label='Nodes in Network (Citing/Cited)')

# 축 1: 엣지수 (Line) - 인용 관계의 총량
ax1.plot(df['연도'], df['엣지수'], color='steelblue', marker='o', linewidth=2, label='Edges (Citations)')

ax1.set_xlabel('Year', fontsize=12)
ax1.set_ylabel('Count (Papers/Citations)', fontsize=12)
ax1.set_xticks(df['연도'])
ax1.legend(loc='upper left')

# 축 2: 밀도 (Secondary Y-axis)
ax2 = ax1.twinx()
ax2.plot(df['연도'], df['밀도'], color='tab:red', marker='s', linestyle='--', label='Network Density')
ax2.set_ylabel('Network Density', color='tab:red', fontsize=12)
ax2.tick_params(axis='y', labelcolor='tab:red')
ax2.legend(loc='upper right')

# 그래프 제목 및 설정
plt.title('Growth and Structural Evolution of AI Law Research (2015-2024)', fontsize=14, pad=20)
plt.grid(axis='y', linestyle='--', alpha=0.7)

# 레이아웃 조정 및 저장
plt.tight_layout()
plt.savefig('AI_law_discrete_trends_updated.png', dpi=300)
plt.show()

###

# 개별 분석 시작

## 발행연도 상관 없이 특정 연도까지 인용된 모든 논문

In [ ]:
import pandas as pd
import networkx as nx
import community.community_louvain as louvain
import os
from tqdm import tqdm
from collections import Counter

# 1. 환경 설정
OUTPUT_DIR = '01_AI_누적_지식_기반_분석'
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

# 2. 데이터 로드
print("데이터를 로드하는 중...")
# 데이터 파일명은 사용자님의 파일 환경에 맞춰 'KCI_AI_전용_논문_노드.csv' 등으로 수정 가능합니다.
nodes = pd.read_csv('KCI_AI_전용_논문_노드.csv')
edges = pd.read_csv('KCI_AI_전용_인용_엣지.csv')

# 엣지에 발행연도 매칭 (Source 논문의 발행연도를 기준으로 누적 범위를 결정)
# 주의: 이 merge는 인용을 '한' 논문의 연도를 붙이는 작업입니다.
edges_with_year = edges.merge(nodes[['node_id', '발행연도']], left_on='source_id', right_on='node_id', how='left')

def run_cumulative_intellectual_base_tracking(start_year=2015, end_year=2024):
    for year in tqdm(range(start_year, end_year + 1), desc="연도별 누적 분석 진행 중"):
        # [핵심 로직] 기준 연도(year) 이하에 발행된 논문들이 생성한 모든 엣지 추출
        # 2015년 이전의 인용 기록이 edges에 있다면, 이 필터링을 통해 과거의 권위가 합산됩니다.
        cum_edges = edges_with_year[edges_with_year['발행연도'] <= year]
        
        if cum_edges.empty:
            continue
            
        # 그래프 생성
        G_dir = nx.DiGraph()
        G_dir.add_edges_from(zip(cum_edges['source_id'], cum_edges['target_id_final']))
        
        # 커뮤니티 탐지 (Louvain은 비방향 그래프 기준)
        G_undir = G_dir.to_undirected()
        partition = louvain.best_partition(G_undir)
        
        # 지표 계산
        comm_counts = Counter(partition.values())
        top_5_communities = [comm[0] for comm in comm_counts.most_common(5)]
        in_degrees = dict(G_dir.in_degree())
        
        year_data = []
        for comm_rank, comm_id in enumerate(top_5_communities, 1):
            # 해당 커뮤니티 소속 노드 추출
            comm_nodes = [n for n, c in partition.items() if c == comm_id]
            
            # 커뮤니티 내 논문들을 피인용수(In-degree) 기준으로 정렬
            comm_papers_sorted = sorted(
                [(n, in_degrees.get(n, 0)) for n in comm_nodes],
                key=lambda x: x[1],
                reverse=True
            )[:5]
            
            for paper_rank, (node_id, citation_count) in enumerate(comm_papers_sorted, 1):
                paper_info = nodes[nodes['node_id'] == node_id]
                if not paper_info.empty:
                    year_data.append({
                        '커뮤니티_순위': comm_rank,
                        '커뮤니티_ID': comm_id,
                        '커뮤니티_크기': comm_counts[comm_id],
                        '논문_순위': paper_rank,
                        '인용수': citation_count,
                        '제목': paper_info['제목'].values[0],
                        '저자': paper_info['저자'].values[0],
                        '발행연도': paper_info['발행연도'].values[0],
                        'node_id': node_id
                    })
        
        # 결과 저장
        if year_data:
            year_df = pd.DataFrame(year_data)
            file_path = os.path.join(OUTPUT_DIR, f"cumulative_{year}.csv")
            year_df.to_csv(file_path, index=False, encoding='utf-8-sig')

if __name__ == "__main__":
    run_cumulative_intellectual_base_tracking()
    print(f"\n✅ 누적 지식 기반 분석 완료! '{OUTPUT_DIR}' 폴더를 확인하세요.")

## 발행연도 상관 없이 특정 연도에 인용된 모든 논문

In [ ]:
import pandas as pd
import networkx as nx
import community.community_louvain as louvain
import os
from tqdm import tqdm
from collections import Counter

# 1. 환경 설정
OUTPUT_DIR = '02_AI_시점별_담론_지형_분석'
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

# 2. 데이터 로드
print("데이터를 로드하는 중...")
nodes = pd.read_csv('KCI_AI_전용_논문_노드.csv')
edges = pd.read_csv('KCI_AI_전용_인용_엣지.csv')

# 엣지에 발행연도 매칭 (인용을 수행한 Source 논문의 발행연도 기준)
edges_with_year = edges.merge(nodes[['node_id', '발행연도']], left_on='source_id', right_on='node_id', how='left')

def run_snapshot_discourse_tracking(start_year=2015, end_year=2024):
    for year in tqdm(range(start_year, end_year + 1), desc="연도별 스냅샷 분석 진행 중"):
        # [핵심 로직] 오직 해당 연도(year)에 발행된 논문들이 생성한 엣지만 추출
        # 이 필터링을 통해 '그해의 연구자들이 선택한 지식'만 남게 됩니다.
        snapshot_edges = edges_with_year[edges_with_year['발행연도'] == year]
        
        if snapshot_edges.empty:
            continue
            
        # 그래프 생성
        G_dir = nx.DiGraph()
        G_dir.add_edges_from(zip(snapshot_edges['source_id'], snapshot_edges['target_id_final']))
        
        # 커뮤니티 탐지 (Louvain)
        G_undir = G_dir.to_undirected()
        partition = louvain.best_partition(G_undir)
        
        # 지표 계산
        comm_counts = Counter(partition.values())
        top_5_communities = [comm[0] for comm in comm_counts.most_common(5)]
        in_degrees = dict(G_dir.in_degree())
        
        year_data = []
        for comm_rank, comm_id in enumerate(top_5_communities, 1):
            comm_nodes = [n for n, c in partition.items() if c == comm_id]
            
            # 당해 연도 피인용수 기준으로 정렬 (해당 시점의 인기 논문 식별)
            comm_papers_sorted = sorted(
                [(n, in_degrees.get(n, 0)) for n in comm_nodes],
                key=lambda x: x[1],
                reverse=True
            )[:5]
            
            for paper_rank, (node_id, citation_count) in enumerate(comm_papers_sorted, 1):
                paper_info = nodes[nodes['node_id'] == node_id]
                if not paper_info.empty:
                    year_data.append({
                        '커뮤니티_순위': comm_rank,
                        '커뮤니티_ID': comm_id,
                        '커뮤니티_크기': comm_counts[comm_id],
                        '논문_순위': paper_rank,
                        '인용수(당해)': citation_count,
                        '제목': paper_info['제목'].values[0],
                        '저자': paper_info['저자'].values[0],
                        '원래_발행연도': paper_info['발행연도'].values[0],
                        'node_id': node_id
                    })
        
        # 결과 저장
        if year_data:
            year_df = pd.DataFrame(year_data)
            file_path = os.path.join(OUTPUT_DIR, f"snapshot_{year}.csv")
            year_df.to_csv(file_path, index=False, encoding='utf-8-sig')

if __name__ == "__main__":
    run_snapshot_discourse_tracking()
    print(f"\n✅ 시점별 담론 분석 완료! '{OUTPUT_DIR}' 폴더를 확인하세요.")

## 특정 연도에 발행한 논문의 중 피인용 횟수 기준

In [ ]:
import pandas as pd
import networkx as nx
import community.community_louvain as louvain
import os
from tqdm import tqdm
from collections import Counter

# 1. 환경 설정
OUTPUT_DIR = '03_AI_연도별_신성_담론_분석'
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

# 2. 데이터 로드
print("데이터를 로드하는 중...")
nodes = pd.read_csv('KCI_AI_전용_논문_노드.csv')
edges = pd.read_csv('KCI_AI_전용_인용_엣지.csv')

def run_rising_star_paper_tracking(start_year=2015, end_year=2024):
    print("전체 기간 통합 네트워크 분석 중 (지식 구조 식별)...")
    # [핵심 로직] 전체 기간의 엣지를 사용하여 '글로벌 위상'과 '글로벌 커뮤니티'를 먼저 계산합니다.
    G_total = nx.DiGraph()
    G_total.add_edges_from(zip(edges['source_id'], edges['target_id_final']))
    
    # 1. 전체 네트워크에서의 피인용수(Global In-degree) 계산
    global_in_degrees = dict(G_total.in_degree())
    
    # 2. 전체 네트워크에서의 커뮤니티 탐지 (지식의 거대한 줄기 파악)
    G_undir = G_total.to_undirected()
    global_partition = louvain.best_partition(G_undir)
    global_comm_counts = Counter(global_partition.values())

    for year in tqdm(range(start_year, end_year + 1), desc="연도별 발행작 분석 진행 중"):
        # [핵심 로직] 오직 해당 연도(year)에 '발행된' 논문들만 추출
        year_born_papers = nodes[nodes['발행연도'] == year].copy()
        
        if year_born_papers.empty:
            continue
            
        # 글로벌 지표 결합
        year_born_papers['인용수_전체'] = year_born_papers['node_id'].map(global_in_degrees).fillna(0)
        year_born_papers['커뮤니티_ID'] = year_born_papers['node_id'].map(global_partition)
        
        # 해당 연도 발행작들이 기여하고 있는 상위 5개 글로벌 커뮤니티 식별
        top_5_comms = year_born_papers['커뮤니티_ID'].value_counts().head(5).index
        
        year_data = []
        for comm_rank, comm_id in enumerate(top_5_comms, 1):
            # 해당 연도 발행 + 특정 커뮤니티 소속 논문 필터링
            comm_papers = year_born_papers[year_born_papers['커뮤니티_ID'] == comm_id]
            
            # 전체 피인용수 기준으로 정렬 (미래의 고전이 될 논문 식별)
            top_5_papers = comm_papers.sort_values(by='인용수_전체', ascending=False).head(5)
            
            for paper_rank, (_, row) in enumerate(top_5_papers.iterrows(), 1):
                year_data.append({
                    '커뮤니티_순위': comm_rank,
                    '커뮤니티_ID': comm_id,
                    '커뮤니티_전체크기': global_comm_counts[comm_id],
                    '논문_순위': paper_rank,
                    '누적_인용수': row['인용수_전체'],
                    '제목': row['제목'],
                    '저자': row['저자'],
                    '발행연도': year, # 파일 내 확인용
                    'node_id': row['node_id']
                })
        
        # 결과 저장
        if year_data:
            year_df = pd.DataFrame(year_data)
            file_path = os.path.join(OUTPUT_DIR, f"rising_stars_{year}.csv")
            year_df.to_csv(file_path, index=False, encoding='utf-8-sig')

if __name__ == "__main__":
    run_rising_star_paper_tracking()
    print(f"\n✅ 연도별 신성 담론 분석 완료! '{OUTPUT_DIR}' 폴더를 확인하세요.")

## 특정 연도에 발행한 논문의 중 피인용 횟수 기준 2년 이내

In [ ]:
import pandas as pd
import networkx as nx
import community.community_louvain as louvain
import os
from tqdm import tqdm
from collections import Counter

# 1. 환경 설정
OUTPUT_DIR = '04_AI_최신_연구_전선_분석'
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

# 2. 데이터 로드
print("데이터를 로드하는 중...")
nodes = pd.read_csv('KCI_AI_전용_논문_노드.csv')
edges = pd.read_csv('KCI_AI_전용_인용_엣지.csv')

# 엣지에 Source(인용한 논문)와 Target(인용된 논문)의 발행연도를 모두 매칭
# 1. Source 연도 매칭
edges_meta = edges.merge(nodes[['node_id', '발행연도']], left_on='source_id', right_on='node_id', how='left')
edges_meta = edges_meta.rename(columns={'발행연도': 'source_year'})

# 2. Target 연도 매칭
edges_meta = edges_meta.merge(nodes[['node_id', '발행연도']], left_on='target_id_final', right_on='node_id', how='left')
edges_meta = edges_meta.rename(columns={'발행연도': 'target_year'})

def run_research_front_tracking(start_year=2015, end_year=2024):
    for year in tqdm(range(start_year, end_year + 1), desc="연구 전선 분석 진행 중"):
        # [핵심 로직] 
        # 1. 인용 대상(Target)이 분석 연도(year)에 발행된 논문이어야 함
        # 2. 인용한 논문(Source)이 발행 연도부터 +1년(year+1) 사이에 발행되어야 함
        mask_front = (edges_meta['target_year'] == year) & \
                     (edges_meta['source_year'] >= year) & \
                     (edges_meta['source_year'] <= year + 1)
        
        front_edges = edges_meta[mask_front]
        
        if front_edges.empty:
            continue
            
        # 그래프 생성
        G_dir = nx.DiGraph()
        G_dir.add_edges_from(zip(front_edges['source_id'], front_edges['target_id_final']))
        
        # 커뮤니티 탐지 (Louvain)
        G_undir = G_dir.to_undirected()
        partition = louvain.best_partition(G_undir)
        
        # 지표 계산
        comm_counts = Counter(partition.values())
        top_5_communities = [comm[0] for comm in comm_counts.most_common(5)]
        in_degrees = dict(G_dir.in_degree())
        
        year_data = []
        for comm_rank, comm_id in enumerate(top_5_communities, 1):
            comm_nodes = [n for n, c in partition.items() if c == comm_id]
            
            # 단기 피인용수 기준으로 정렬 (초기 파급력이 강한 논문 식별)
            comm_papers_sorted = sorted(
                [(n, in_degrees.get(n, 0)) for n in comm_nodes],
                key=lambda x: x[1],
                reverse=True
            )[:5]
            
            for paper_rank, (node_id, citation_count) in enumerate(comm_papers_sorted, 1):
                paper_info = nodes[nodes['node_id'] == node_id]
                if not paper_info.empty:
                    year_data.append({
                        '커뮤니티_순위': comm_rank,
                        '커뮤니티_ID': comm_id,
                        '커뮤니티_크기': comm_counts[comm_id],
                        '논문_순위': paper_rank,
                        '단기_인용수(2년)': citation_count,
                        '제목': paper_info['제목'].values[0],
                        '저자': paper_info['저자'].values[0],
                        '발행연도': year,
                        'node_id': node_id
                    })
        
        # 결과 저장
        if year_data:
            year_df = pd.DataFrame(year_data)
            file_path = os.path.join(OUTPUT_DIR, f"research_front_{year}.csv")
            year_df.to_csv(file_path, index=False, encoding='utf-8-sig')

if __name__ == "__main__":
    run_research_front_tracking()
    print(f"\n✅ 최신 연구 전선 분석 완료! '{OUTPUT_DIR}' 폴더를 확인하세요.")

### 엑셀 컴바인

In [5]:
import pandas as pd
import os

# 1. CSV 파일들이 있는 폴더 경로 설정
folder_path = './04_AI_최신_연구_전선_분석'  # CSV 파일이 저장된 폴더명으로 수정하세요
output_file = '04_AI_최신_연구_전선_분석결과.xlsx'

# 2. ExcelWriter 객체 생성
with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    # 폴더 내의 모든 파일 확인
    for file_name in os.listdir(folder_path):
        if file_name.endswith('.csv'):
            # 파일 경로 생성
            file_path = os.path.join(folder_path, file_name)
            
            # CSV 읽기 (인코딩 문제는 필요시 encoding='cp949' 추가)
            df = pd.read_csv(file_path)
            
            # 시트 이름 설정 (파일명에서 .csv 제외, 최대 31자 제한 준수)
            sheet_name = file_name[:-4][:31]
            
            # 엑셀 시트로 쓰기
            df.to_excel(writer, sheet_name=sheet_name, index=False)

print(f"합치기 완료! 생성된 파일: {output_file}")

합치기 완료! 생성된 파일: 04_AI_최신_연구_전선_분석결과.xlsx


### MPA 분석

In [ ]:
import pandas as pd
import networkx as nx
import time

# 1. 데이터 로드
print("1/5. 데이터 로딩 중...")
nodes = pd.read_csv('KCI_AI_전용_논문_노드.csv')
edges = pd.read_csv('KCI_AI_전용_인용_엣지.csv')

# --- [추가] 연도 범위 설정 라인 ---
start_year = 2015  # 시작 연도
end_year = 2024    # 종료 연도

# 설정한 범위에 해당하는 노드만 필터링
nodes = nodes[(nodes['발행연도'] >= start_year) & (nodes['발행연도'] <= end_year)]

# 필터링된 노드들에 속한 엣지만 남기기
valid_node_ids = set(nodes['node_id'])
edges = edges[edges['source_id'].isin(valid_node_ids) & edges['target_id_final'].isin(valid_node_ids)]

# 2. 방향성 그래프 생성
G = nx.DiGraph()
G.add_edges_from(zip(edges['source_id'], edges['target_id_final']))

# [수정] 사이클 제거 로직 강화
if not nx.is_directed_acyclic_graph(G):
    print("⚠️ 사이클(Cycle)이 감지되어 제거를 시작합니다.")
    while not nx.is_directed_acyclic_graph(G):
        try:
            # 사이클을 하나 찾아 그 중 첫 번째 엣지를 제거
            cycle = nx.find_cycle(G, orientation='original')
            G.remove_edge(cycle[0][0], cycle[0][1])
        except nx.NetworkXNoCycle:
            break
    print("✅ 모든 사이클 제거 완료.")

def calculate_spc_optimized(G):
    """위상 정렬을 이용한 O(V+E) SPC 가중치 계산"""
    topo_order = list(nx.topological_sort(G))
    
    f = {n: 0 for n in G.nodes()}
    for n in topo_order:
        if G.in_degree(n) == 0:
            f[n] = 1
        for successor in G.successors(n):
            f[successor] += f[n]
            
    b = {n: 0 for n in G.nodes()}
    for n in reversed(topo_order):
        if G.out_degree(n) == 0:
            b[n] = 1
        for predecessor in G.predecessors(n):
            b[predecessor] += b[n]
            
    edge_spc = {(u, v): f[u] * b[v] for u, v in G.edges()}
    return edge_spc

def extract_main_path(G, edge_weights):
    if not edge_weights: return []
    
    # 1. 최고 가중치 엣지 탐색
    max_e = max(edge_weights, key=edge_weights.get)
    path = [max_e]
    
    # 2. Forward (Sink 방향)
    curr = max_e[1]
    while G.out_degree(curr) > 0:
        out_edges = list(G.out_edges(curr))
        # 가중치가 있는 엣지만 선별하여 그 중 최대값 선택
        valid_out_edges = [e for e in out_edges if e in edge_weights]
        if not valid_out_edges: break
        nxt_e = max(valid_out_edges, key=lambda e: edge_weights[e])
        path.append(nxt_e)
        curr = nxt_e[1]
        
    # 3. Backward (Source 방향)
    curr = max_e[0]
    while G.in_degree(curr) > 0:
        in_edges = list(G.in_edges(curr))
        valid_in_edges = [e for e in in_edges if e in edge_weights]
        if not valid_in_edges: break
        prev_e = max(valid_in_edges, key=lambda e: edge_weights[e])
        path.insert(0, prev_e)
        curr = prev_e[0]
        
    return path

# 3. 실행
start_time = time.time()
print("2/5. SPC 가중치 계산 중...")
edge_weights = calculate_spc_optimized(G)

print("3/5. 주경로 추출 중...")
main_path_edges = extract_main_path(G, edge_weights)

# 4. 결과 정리
path_node_ids = []
for u, v in main_path_edges:
    if u not in path_node_ids: path_node_ids.append(u)
    if v not in path_node_ids: path_node_ids.append(v)

main_path_info = pd.DataFrame({'node_id': path_node_ids}).merge(
    nodes[['node_id', '제목', '저자', '발행연도']], on='node_id', how='left'
)

print(f"\n✅ 분석 완료! (소요시간: {time.time() - start_time:.2f}초)")
print("\n🏆 추출된 주경로 논문 리스트:")
print(main_path_info[['발행연도', '제목', '저자']])

# 5. 저장
main_path_info.to_csv('AI법학_최종_주경로_결과.csv', index=False, encoding='utf-8-sig')